In [1]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [2]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

# === 예제 리스트 정의 ===
# 각 딕셔너리가 입출력 예시 1건(1 shot)입니다.
# 여기서는 기술 용어를 한 줄로 설명하는 패턴을 보여줍니다.
examples = [
    {"term": "RAG", "definition": "외부 문서를 검색하여 LLM 입력에 삽입한 뒤 답변을 생성하는 구조"},
    {"term": "임베딩", "definition": "텍스트를 고정 길이의 숫자 벡터로 변환하는 방법"},
    {"term": "토큰", "definition": "LLM이 텍스트를 처리할 때 사용하는 최소 단위"},
]

In [3]:
# === 각 예시를 어떤 형식으로 표현할지 정의 ===
# 예시 리스트의 각 딕셔너리가 이 템플릿에 주입됩니다.
example_prompt = PromptTemplate(
    input_variables=["term", "definition"],
    template="용어: {term}\n정의: {definition}"
)

# === 하나의 예시가 어떻게 변환되는지 확인 ===
print(example_prompt.format(term="RAG", definition="외부 문서를 검색하여 LLM 입력에 삽입한 뒤 답변을 생성하는 구조"))

용어: RAG
정의: 외부 문서를 검색하여 LLM 입력에 삽입한 뒤 답변을 생성하는 구조


In [4]:
# === FewShotPromptTemplate 구성 ===
prompt = FewShotPromptTemplate(
    examples=examples,           # 예시 리스트
    example_prompt=example_prompt, # 각 예시의 포맷 틀
    prefix="다음 예시를 참고하여, 주어진 기술 용어를 같은 형식으로 정의하세요.",  # 예시 앞에 붙는 작업 설명
    suffix="용어: {term}\n정의:",  # 예시 뒤에 붙는 실제 입력 자리
    input_variables=["term"]       # 실행 시 주입할 변수
)

# === 완성된 프롬프트 확인 ===
result = prompt.format(term="LCEL")
print(result)

다음 예시를 참고하여, 주어진 기술 용어를 같은 형식으로 정의하세요.

용어: RAG
정의: 외부 문서를 검색하여 LLM 입력에 삽입한 뒤 답변을 생성하는 구조

용어: 임베딩
정의: 텍스트를 고정 길이의 숫자 벡터로 변환하는 방법

용어: 토큰
정의: LLM이 텍스트를 처리할 때 사용하는 최소 단위

용어: LCEL
정의:


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# === Chat Model 생성 ===
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY
)
parser = StrOutputParser()

# === LCEL 체인 구성 및 실행 ===
chain = prompt | model | parser

result = chain.invoke({"term": "LCEL"})
print(f"용어: LCEL")
print(f"정의: {result}")

용어: LCEL
정의: 용어: LCEL
정의: LangChain에서 구성 요소를 연결하고 오케스트레이션하기 위한 표현 언어


In [8]:
# === 같은 템플릿, 다른 입력 ===
terms = ["ChatPromptTemplate", "벡터 데이터베이스", "프롬프트 엔지니어링"]

for t in terms:
    result = chain.invoke({"term": t})
    print(f"용어: {t}")
    print(f"정의: {result}")
    print()

용어: ChatPromptTemplate
정의: 용어: ChatPromptTemplate
정의: 채팅 모델의 입력으로 사용될 여러 메시지를 구조화하여 정의하는 템플릿 형식

용어: 벡터 데이터베이스
정의: 용어: 벡터 데이터베이스
정의: 고차원 벡터 데이터를 저장하고 유사성 검색을 효율적으로 수행하도록 특화된 데이터베이스

용어: 프롬프트 엔지니어링
정의: 정의: LLM이 원하는 답변을 생성하도록 프롬프트를 설계하고 최적화하는 기술

